# Linear regression baseline

Per-station linear regression on ALADIN precipitation, station rainfall and recent discharge, evaluated on the same yearly splits as the TOK model (test years 2021, 2022, 2023). Results are logged to the W&B project `Final Linear Regression`, which `performance_graphs.ipynb` compares against.

In [ ]:
import importlib
import hidro_and_meteo_helper_functions as sf  # station functions
importlib.reload(sf)

In [ ]:
###
### READ HIDROLOGICAL DATA FOR ONE STATION
###
postaja = 4200

sf.get_Q_df_for_station(postaja)

In [ ]:
###
### READ METEOROLOGICAL DATA FOR ONE STATION
###
sf.get_meteo_df_for_station(2210)
sf.get_meteo_station_loc(2210)
sf.get_k_closest_meteo_stations(4200, 3)

In [ ]:
sf.get_station_and_river_name(4200)
print(sf.get_station_and_river_name(4200))

In [ ]:
# Load Aladin precipitation data
import xarray as xr
#xr.open_dataset('data/ALADIN/aladin_tp_regridded.nc', decode_timedelta=True)

In [ ]:
import numpy as np
# hardcoded aladin raw latitudes and longitudes
aladin_raw_latitude = np.array([45.202, 45.242, 45.282, 45.322, 45.362, 45.402, 45.442, 45.482, 45.522,
       45.562, 45.602, 45.642, 45.682, 45.722, 45.762, 45.802, 45.842, 45.882,
       45.922, 45.962, 46.002, 46.042, 46.082, 46.122, 46.162, 46.202, 46.242,
       46.282, 46.322, 46.362, 46.402, 46.442, 46.482, 46.522, 46.562, 46.602,
       46.642, 46.682, 46.722, 46.762, 46.802, 46.842, 46.882, 46.922, 46.962,
       47.002, 47.042])
aladin_raw_longitude = np.array([13.05 , 13.107, 13.164, 13.221, 13.278, 13.335, 13.392, 13.449, 13.506,
       13.563, 13.62 , 13.677, 13.734, 13.791, 13.848, 13.905, 13.962, 14.019,
       14.076, 14.133, 14.19 , 14.247, 14.304, 14.361, 14.418, 14.475, 14.532,
       14.589, 14.646, 14.703, 14.76 , 14.817, 14.874, 14.931, 14.988, 15.045,
       15.102, 15.159, 15.216, 15.273, 15.33 , 15.387, 15.444, 15.501, 15.558,
       15.615, 15.672, 15.729, 15.786, 15.843, 15.9  , 15.957, 16.014, 16.071,
       16.128, 16.185, 16.242, 16.299, 16.356, 16.413, 16.47 , 16.527, 16.584,
       16.641, 16.698, 16.755, 16.812, 16.869, 16.926, 16.983, 17.04 ])

In [ ]:
from torch.utils.data import Dataset
class CombinedDataset(Dataset):
    def __init__(self, dataset1, dataset2):
        self.dataset1 = dataset1
        self.dataset2 = dataset2

    def __len__(self):
        return len(self.dataset1) + len(self.dataset2)
    
    def __getitem__(self, idx):
        if idx < len(self.dataset1):
            return self.dataset1[idx]
        else:
            return self.dataset2[idx - len(self.dataset1)]

In [ ]:
import torch
import yaml
from torch.utils.data import Dataset, DataLoader
import cfgrib
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import json
from tqdm import tqdm
import logging
from sklearn.preprocessing import MinMaxScaler

# Configure self.self.self.logger

logger = logging.getLogger('data_linreg')
logger.setLevel(logging.INFO)
if not logger.hasHandlers():
    handler = logging.StreamHandler()
    formatter = logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s')
    handler.setFormatter(formatter)
    logger.addHandler(handler)

class DATA_linreg(Dataset):
    
    def model_data_load(self):
        long, lat = sf.get_hidro_station_loc(self.hidro_station_id)

        # find closest longitude and latitude
        lon_idx = np.abs(aladin_raw_longitude - long).argmin()
        lat_idx = np.abs(aladin_raw_latitude - lat).argmin()

        # take (5,5) area around it
        min_lat = lat_idx - self.expand_lat_lon_area[0]
        max_lat = lat_idx + self.expand_lat_lon_area[0] + 1
        min_lon = lon_idx - self.expand_lat_lon_area[1]
        max_lon = lon_idx + self.expand_lat_lon_area[1] + 1

        self.ALADIN = xr.open_dataset(f"data/ALADIN/aladin_merged_ds.nc",
                                      decode_timedelta=True
                                      ).isel(lat=slice(min_lat, max_lat),
                                            lon=slice(min_lon, max_lon),
                                            step=slice(0, 24)#self.prediction_hours)
                                            )[['tp']].compute()
        self.ALADIN = self.ALADIN.sel(time=slice(f"{self.from_year}-01-01", f"{self.up_to_year}-01-01"))
        self.ALADIN = self.ALADIN.drop_duplicates('time')

        if 'timedelta' not in str(self.ALADIN.step.dtype):
            self.ALADIN = self.ALADIN.assign_coords(step=pd.to_timedelta(self.ALADIN.step, unit='h'))
        self.DATA = self.ALADIN

        self.DATA = self.DATA.sortby('time')
        
        return self.DATA
    
    def meteo_data_load(self):
        self.meteo_data = sf.get_k_closest_meteo_stations(self.hidro_station_id, 3)
        return self.meteo_data

    def __init__(self, station_id, from_year, up_to_year,
                  meteo_past_days: int = None, flow_past_hours: int = None,
                  RUN_NAME: str = None, prediction_hours: int = 18, expand_lat_lon_area: tuple = (0, 0),
                  reduce_features_over_lat_lon: bool = False,
                  generate_extra_data_points: bool = False,
                  run_parralel: bool = True,
                  predefined_scalers = None,
                  ):
        
        self.hidro_station_id = station_id
        self.from_year = from_year
        self.up_to_year = up_to_year
        self.prediction_hours = prediction_hours
        self.expand_lat_lon_area = expand_lat_lon_area
        self.flow_past_hours = flow_past_hours
        self.reduce_features_over_lat_lon = reduce_features_over_lat_lon # if true features wont be kept for each lat and lon, but will be summed over the whole area
        self.generate_extra_data_points = generate_extra_data_points
        self.logger = logger

        if self.expand_lat_lon_area != (0,0) and not self.reduce_features_over_lat_lon:
            self.logger.info("Features are kept for each lat and lon, if you want to reduce features over lat and lon, set reduce_features_over_lat_lon to True")

        # load data in ok format -> 
        self.meteo_data = self.meteo_data_load()
        self.DATA = self.model_data_load()
        self.station_Q = sf.get_Q_df_for_station(self.hidro_station_id).sort_index()

        #normalize data
        if predefined_scalers is None:
            self.scalers = {}
            self.scalers['tp'] = MinMaxScaler().fit(self.DATA.tp.values.reshape(-1,1))
            self.DATA.tp.values = self.scalers['tp'].transform(self.DATA.tp.values.reshape(-1,1)).reshape(self.DATA.tp.values.shape)
        else:
            self.scalers = predefined_scalers
            self.DATA.tp.values = self.scalers['tp'].transform(self.DATA.tp.values.reshape(-1,1)).reshape(self.DATA.tp.values.shape)

        # training example is defined by hour at which we want to predict
        potential_items = self.DATA.time.values
        
        # check if we can create a training example for each time, and save training examples into a list
        self.valid_items = []
        
        # Define hour range based on whether we want to generate extra data points
        hour_range = range(8) if self.generate_extra_data_points else [7]
        
        from multiprocess import Pool
        
        def process_item(args):
            import numpy as np
            self_obj, date, hour_offset = args
            item = self_obj.get_item_by_date_and_prediction_hour_offset(date, hour_offset)
            if item is not None and not np.isnan(item['features']).any() and not np.isnan(item['target']).any():
                return item
            else:
                if item is None:
                    self.logger.debug(f"Invalid item: {date}")
                elif np.isnan(item['features']).any():
                    self.logger.debug(f"Invalid item: {date}, some feature is nan")
                elif np.isnan(item['target']).any():
                    self.logger.debug(f"Invalid item: {date}, some target is nan")
                else:
                    self.logger.debug(f"Invalid item: {date}, some other error")
                return None
        
        items_to_process = [(self, date, hour) for date in potential_items for hour in hour_range]
        if run_parralel:
            num_processes = 8
            with Pool(num_processes) as pool:
                results = list(tqdm(pool.imap(process_item, items_to_process, chunksize = min(1000,len(items_to_process)//num_processes+1)), 
                                total=len(items_to_process), 
                                desc="Checking valid items"))
        else:
            results = [process_item(item) for item in items_to_process]
        
        self.valid_items = [item for item in results if item is not None]
        
        self.logger.info(f"Tested {len(potential_items)*len(hour_range)} items, {len(self.valid_items)} are valid, which is {len(self.valid_items) / (len(potential_items)*len(hour_range)) * 100:.2f}%")

    def __getitem__(self, idx):
        return self.valid_items[idx]
    
    def __len__(self):
        return len(self.valid_items)

    def get_model_data_values(self, date, prediction_hour_offset = 7):
        import pandas as pd
        import numpy as np
        try:
            prediction_day_data = self.DATA.sel(time=date).isel(step=slice(0, 24+min(0, prediction_hour_offset-7))).tp.values
        except KeyError:
            self.logger.debug(f'No prediction day data for {date}')
            return None
        try:
            previous_day_data = self.DATA.sel(time=date - pd.Timedelta(days=1)).isel(step=slice(0+max(0,prediction_hour_offset-7), 24)).tp.values
        except KeyError:
            self.logger.debug(f'No previous day data for {date}')
            return None
        model_data_values = np.concatenate([previous_day_data, prediction_day_data], axis=0)
        model_data_values = model_data_values.reshape(-1, *model_data_values.shape[-2:]) # flatten time and step keep lat and lon
        if prediction_hour_offset > 7:
            try:
                next_day_data = self.DATA.sel(time=date + pd.Timedelta(days=1)).isel(step=slice(0, prediction_hour_offset-7)).tp.values
            except KeyError:
                self.logger.debug(f'No next day data for {date}')
                return None
            next_day_data = next_day_data.reshape(-1, *next_day_data.shape[-2:]) # flatten time and step keep lat and lon
            model_data_values = np.concatenate([model_data_values, next_day_data], axis=0)
        elif prediction_hour_offset < 7:
            # Take data from 2 days before for offsets < 7
            try:
                two_days_before = self.DATA.sel(time=date - pd.Timedelta(days=2)).isel(step=slice(24+(prediction_hour_offset-7), 24)).tp.values
            except KeyError:
                self.logger.debug(f'No two days before data for {date}')
                return None
            two_days_before = two_days_before.reshape(-1, *two_days_before.shape[-2:]) # flatten time and step keep lat and lon
            model_data_values = np.concatenate([two_days_before, model_data_values], axis=0)    

        _, lat, lon = model_data_values.shape
        if model_data_values.shape != (48, lat, lon):
            self.logger.debug(f'Model data values shape is not (48, lat, lon), but {model_data_values.shape}')
            return None

        return model_data_values

    def get_item_by_date_and_prediction_hour_offset(self, date, prediction_hour_offset = 7):
        import pandas as pd
        import numpy as np
        """
        Generate item by date and prediction hour offset. Default prediction hour offset is 7 hours, which is what for what time arso predicts. 
        But we can generate more data by also taking into account data around that time
        """

        model_data_values = self.get_model_data_values(date, prediction_hour_offset)
        if model_data_values is None:
            return None
        past_model_data = model_data_values[:24, :, :]
        future_model_data = model_data_values[24:, :, :]

        # instead of keeping every lat and lon, we sum over them with one value
        if self.reduce_features_over_lat_lon:
            past_model_data = past_model_data.sum(axis=(-1,-2)).reshape(-1,1,1)
            future_model_data = future_model_data.sum(axis=(-1,-2)).reshape(-1,1,1)

        # flow we want to predict - date is at 00:00, to match arso predictions, first prediction is at 07:00
        first_prediction_hour = date + pd.Timedelta(hours=prediction_hour_offset)
        future_Q = self.station_Q.loc[first_prediction_hour: first_prediction_hour + pd.Timedelta(hours=self.prediction_hours-1)]
        # flow for past hours
        past_Q = self.station_Q.loc[first_prediction_hour - pd.Timedelta(hours=self.flow_past_hours): first_prediction_hour - pd.Timedelta(hours=1)]
        mean_past_24h_Q = self.station_Q.loc[first_prediction_hour - pd.Timedelta(hours=24): first_prediction_hour - pd.Timedelta(hours=1)].mean()
        mean_past_48h_Q = self.station_Q.loc[first_prediction_hour - pd.Timedelta(hours=48): first_prediction_hour - pd.Timedelta(hours=1)].mean()
        mean_past_week_Q = self.station_Q.loc[first_prediction_hour - pd.Timedelta(hours=24*7): first_prediction_hour - pd.Timedelta(hours=1)].mean()

        # meteo data features
        meteo_data_past_24h = self.meteo_data.loc[first_prediction_hour - pd.Timedelta(hours=24): first_prediction_hour- pd.Timedelta(hours=1)]
        if meteo_data_past_24h.isna().values.any() or meteo_data_past_24h.shape[0] != 24:
            self.logger.info(f"Meteo data past 24h is not the same shape as 24, {meteo_data_past_24h.shape}")
            return None
        meteo_past_6h_blocks_rainfall = self.feature_from_meteo_data(meteo_data_past_24h.values, 6)
        mean_past_72h_rainfall = self.meteo_data.loc[first_prediction_hour - pd.Timedelta(hours=72): first_prediction_hour- pd.Timedelta(hours=1)].mean(axis=0).values
        mean_past_48h_rainfall = self.meteo_data.loc[first_prediction_hour - pd.Timedelta(hours=48): first_prediction_hour- pd.Timedelta(hours=1)].mean(axis=0).values
        #print(meteo_data_past_24h.shape)
        
        # flatten the data
        future_Q = future_Q.values.flatten()
        past_Q = past_Q.values.flatten()

        if len(future_Q) != self.prediction_hours or len(past_Q) != self.flow_past_hours:
            if len(future_Q) != self.prediction_hours:
                self.logger.debug('future Q is not the same length as prediction hours')
            if len(past_Q) != self.flow_past_hours:
                self.logger.debug('past Q is not the same length as flow past hours')
            return None

        # future model rainfall in 3h and 6h blocks
        blocks_3h_future_rainfall  = self.feature_xh_blocks(future_model_data, 3)
        blocks_6h_future_rainfall  = self.feature_xh_blocks(future_model_data, 6)
        # past model rainfall in 3h and 6h blocks
        blocks_6h_past_rainfall  = self.feature_xh_blocks(past_model_data, 6)
        blocks_24h_past_rainfall  = self.feature_xh_blocks(past_model_data, 24)



        # stack vectors into long vector
        feature_vector = np.hstack((past_model_data.flatten(),
                                    past_Q, 
                                    future_model_data.flatten(),
                                    blocks_3h_future_rainfall.flatten(),
                                    blocks_6h_future_rainfall.flatten(),
                                    blocks_6h_past_rainfall.flatten(),
                                    blocks_24h_past_rainfall.flatten(),
                                    mean_past_24h_Q,
                                    mean_past_48h_Q,
                                    mean_past_week_Q,
                                    mean_past_48h_rainfall.flatten(),
                                    mean_past_72h_rainfall.flatten(),
                                    meteo_past_6h_blocks_rainfall.flatten()
                                    ))
        return {'target': future_Q, 'features': feature_vector, 'prediction_timestamp': pd.Timestamp(first_prediction_hour - pd.DateOffset(hours=1)).timestamp()}

    def feature_xh_blocks(self, data: np.ndarray, x: int):
        import numpy as np
        # input data is a 4D array with shape - (time, step, lat, lon)
        # we want to split it into x-hour blocks
        if len(data.shape) == 3:  # (time=24, lat, lon)
            time, lat_size, lon_size = data.shape
            n_blocks = time // x
            
            # Reshape to group x consecutive time steps together
            reshaped = data[:n_blocks * x].reshape(n_blocks, x, lat_size, lon_size)
            
            summed_blocks = np.sum(reshaped, axis=1) # Result: (n_blocks, lat, lon)
            
            return summed_blocks
        else:
            # Handle case where data is not 4D
            self.logger.debug(f"Expected 4D data but got shape {data.shape}")
            raise ValueError(f"Expected 4D data but got shape {data.shape}")

    def feature_from_meteo_data(self, data: np.ndarray, x: int):
        import numpy as np
        """
        Create blocks of x hours from meteorological data.
        
        Args:
            data (np.ndarray): Input data array with shape (time, step, lat, lon) or (time, step)
            x (int): Number of hours per block
            
        Returns:
            np.ndarray: Array with summed blocks
        """
        if len(data.shape) == 2:  # (time, stations)
            time, num_stations = data.shape
            n_blocks = time // x
            
            # Reshape to group x consecutive time steps together
            reshaped = data[:n_blocks * x, :].reshape(n_blocks, x, num_stations)
            
            # Sum over the x time steps in each block
            summed_blocks = np.sum(reshaped, axis=1)  # Result: (n_blocks, stations)
            
        else:
            self.logger.debug(f"Expected 2D data but got shape {data.shape}")
            return data
            
        return summed_blocks

    def get_scalers(self):
        return self.scalers.copy()

dd = DATA_linreg(4200, from_year=2020, up_to_year=2021, flow_past_hours=1, prediction_hours=18, expand_lat_lon_area=(1, 1), reduce_features_over_lat_lon=True, generate_extra_data_points=False, run_parralel=True)

In [ ]:
dd.__getitem__(0)['features'].shape, len(dd)
#18 28 36 37]

In [ ]:
station_id = 4200
min_date, max_date = sf.get_Q_df_for_station(station_id).index.min().year,sf.get_Q_df_for_station(station_id).index.max().year
print(f"station min date: {min_date}, max date: {max_date}")

dataset_train_up_to_2021 = DATA_linreg(station_id, min_date, up_to_year=2021, flow_past_hours=1, prediction_hours=18, generate_extra_data_points=True)
dataset_test_2021 = DATA_linreg(station_id, 2021, up_to_year=2022, flow_past_hours=1, prediction_hours=18, generate_extra_data_points=False, predefined_scalers=dataset_train_up_to_2021.get_scalers())
dataset_train_up_to_2022 = DATA_linreg(station_id, min_date, up_to_year=2022, flow_past_hours=1, prediction_hours=18, generate_extra_data_points=True)
dataset_test_2022 = DATA_linreg(station_id, 2022, up_to_year=2023, flow_past_hours=1, prediction_hours=18, generate_extra_data_points=False, predefined_scalers=dataset_train_up_to_2022.get_scalers())
dataset_train_up_to_2023 = DATA_linreg(station_id, min_date, up_to_year=2023, flow_past_hours=1, prediction_hours=18, generate_extra_data_points=True)
dataset_test_2023 = DATA_linreg(station_id, 2023, up_to_year=2024, flow_past_hours=1, prediction_hours=18, generate_extra_data_points=False, predefined_scalers=dataset_train_up_to_2023.get_scalers())
print(dataset_train_up_to_2021.__getitem__(0)['features'].shape)

In [ ]:
cv_splits = [(dataset_train_up_to_2021, dataset_test_2021),
             (dataset_train_up_to_2022, dataset_test_2022),
             (dataset_train_up_to_2023, dataset_test_2023)]

In [ ]:
def prep_cv_splits(station_id):
    min_date, max_date = sf.get_Q_df_for_station(station_id).index.min().year,sf.get_Q_df_for_station(station_id).index.max().year
    print(f"station min date: {min_date}, max date: {max_date}")

    dataset_train_up_to_2021 = DATA_linreg(station_id, min_date, up_to_year=2021, flow_past_hours=1, prediction_hours=18, generate_extra_data_points=True)
    dataset_test_2021 = DATA_linreg(station_id, 2021, up_to_year=2022, flow_past_hours=1, prediction_hours=18, generate_extra_data_points=False, predefined_scalers=dataset_train_up_to_2021.get_scalers())
    dataset_train_up_to_2022 = DATA_linreg(station_id, min_date, up_to_year=2022, flow_past_hours=1, prediction_hours=18, generate_extra_data_points=True)
    dataset_test_2022 = DATA_linreg(station_id, 2022, up_to_year=2023, flow_past_hours=1, prediction_hours=18, generate_extra_data_points=False, predefined_scalers=dataset_train_up_to_2022.get_scalers())
    dataset_train_up_to_2023 = DATA_linreg(station_id, min_date, up_to_year=2023, flow_past_hours=1, prediction_hours=18, generate_extra_data_points=True)
    dataset_test_2023 = DATA_linreg(station_id, 2023, up_to_year=2024, flow_past_hours=1, prediction_hours=18, generate_extra_data_points=False, predefined_scalers=dataset_train_up_to_2023.get_scalers())
    print(dataset_train_up_to_2021.__getitem__(0)['features'].shape)

    cv_splits = [(dataset_train_up_to_2021, dataset_test_2021),
             (dataset_train_up_to_2022, dataset_test_2022),
             (dataset_train_up_to_2023, dataset_test_2023)]
    return cv_splits


In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.pipeline import Pipeline
import importlib
import numpy as np
import plotting_and_evaluation_functions as pef
importlib.reload(pef)
import matplotlib.pyplot as plt
import wandb
import joblib
import os
from sklearn.feature_selection import SequentialFeatureSelector , f_regression
from sklearn.multioutput import MultiOutputRegressor

# Create a custom feature selector that works with multi-output targets
class MultiOutputSelectKBest:
    def __init__(self, k=10):
        self.k = k
        self.selectors = []
        self.selected_features = None
    
    def fit(self, X, y):
        # For multi-output, we'll select features based on the first output
        # or use a different strategy like selecting features that are good for any output
        selector = SequentialFeatureSelector(score_func=f_regression, k=self.k)
        selector.fit(X, y[:, 0])  # Use first output for feature selection
        self.selected_features = selector.get_support()
        return self
    
    def transform(self, X):
        return X[:, self.selected_features]
    
    def fit_transform(self, X, y):
        return self.fit(X, y).transform(X)

regressor = MultiOutputRegressor(LinearRegression())
feature_selector = SequentialFeatureSelector(regressor)

pipeline = Pipeline([
    #('scaler', StandardScaler()), ofc this has no effect
    #('feature_selection', feature_selector),
    ('regressor', regressor)
])


def log_and_evaluate_pipeline(station_id:int, pipeline):

    # Initialize wandb
    run = wandb.init(project=f"Final Linear Regression", name=f"station-{station_id}", settings={"silent": True})
    # Log station name/ID for better tracking in wandb
    station_name, river_name = sf.get_station_and_river_name(station_id)
    wandb.config.update({
        "station_id": station_id,
        "station_name": station_name,
        "river_name": river_name,
        "method": f'linear-regression-station-{station_id}'
    })

    cv_train_losses = []
    cv_test_losses = []
    models = []
    all_predictions = []
    all_actuals = []
    all_timestamps = []
    predictions_dfs = []
    ground_truth_dfs = []

    cv_splits = prep_cv_splits(station_id)

    for i, (train_dataset, test_dataset) in enumerate(cv_splits):
        dataloader_train = DataLoader(train_dataset, batch_size=len(train_dataset))
        dataloader_test = DataLoader(test_dataset, batch_size=len(test_dataset), shuffle=False)

        # get first batch from dataloader
        batch = next(iter(dataloader_train))
        X_train, y_train, prediction_timestamp_train = batch['features'].numpy(), batch['target'].numpy(), list(pd.to_datetime(batch['prediction_timestamp'].numpy(), unit='s'))

        batch = next(iter(dataloader_test))
        X_test, y_test, prediction_timestamp_test = batch['features'].numpy(), batch['target'].numpy(), list(pd.to_datetime(batch['prediction_timestamp'].numpy(), unit='s'))

        # Check that test dates are sorted by date
        # Convert timestamps to numpy array for easier comparison
        prediction_timestamps_array = np.array(prediction_timestamp_test)

        # Check if timestamps are in ascending order
        if not np.all(np.diff(prediction_timestamps_array) >= pd.to_timedelta(0)):
            raise ValueError("Test dates are not sorted by date.")
        
        #print("Data shapes ",X_train.shape, y_train.shape, X_test.shape, y_test.shape)

        pipeline.fit(X_train, y_train)
        
        train_pred = pipeline.predict(X_train)
        train_loss = mean_squared_error(y_train, train_pred)
        print(f"Train MSE: {train_loss:.4f}")
        cv_train_losses.append(train_loss)

        y_pred = pipeline.predict(X_test)
        test_loss = mean_squared_error(y_test, y_pred)
        print(f"Test MSE: {test_loss}")
        cv_test_losses.append(test_loss)
        
        # Save model and predictions for this fold
        models.append(pipeline)
        all_predictions.append(y_pred)
        all_actuals.append(y_test)
        all_timestamps.extend(prediction_timestamp_test)
        
        
        # Save model to file and log as artifact
        model_path = f"model_fold_{i+1}.joblib"
        joblib.dump(pipeline, model_path)
        model_artifact = wandb.Artifact(f"model_fold_{i+1}", type="model")
        model_artifact.add_file(model_path)
        wandb.log_artifact(model_artifact)
        os.remove(model_path)
        
        prediction_df = pd.DataFrame(y_pred, index=pd.Series(prediction_timestamp_test, name='timestamp'))
        predictions_dfs.append(prediction_df)
        ground_truth_df = pd.DataFrame(y_test, index=pd.Series(prediction_timestamp_test, name='timestamp'))
        ground_truth_dfs.append(ground_truth_df)


    mean_train_loss = np.mean(cv_train_losses)
    mean_test_loss = np.mean(cv_test_losses)
    print(f"CV Train Loss: {mean_train_loss:.4f}")
    print(f"CV Test Loss: {mean_test_loss:.4f}")

    # Log average metrics
    wandb.log({
        "mean_train_mse": mean_train_loss,
        "mean_test_mse": mean_test_loss,
        'train_mse': wandb.Table(dataframe = pd.DataFrame(cv_train_losses, columns=['train_mse'])),
        'test_mse': wandb.Table(dataframe = pd.DataFrame(cv_test_losses, columns=['test_mse'])) ,
        'predictions': wandb.Table(dataframe = pd.concat(predictions_dfs))
    })

    # combine all predictions and ground truth
    predictions_df = pd.concat(predictions_dfs)
    ground_truth_df = pd.concat(ground_truth_dfs)

    # Save predictions and ground truth as parquet files
    predictions_path = "predictions.parquet"
    ground_truth_path = "ground_truth.parquet"
    predictions_df.to_parquet(predictions_path)
    ground_truth_df.to_parquet(ground_truth_path)
    # log predictions and ground truth as artifacts
    predictions_artifact = wandb.Artifact(name="predictions", type="dataset")
    predictions_artifact.add_file(predictions_path)
    wandb.log_artifact(predictions_artifact)

    ground_truth_artifact = wandb.Artifact(name="ground_truth", type="dataset")
    ground_truth_artifact.add_file(ground_truth_path)
    wandb.log_artifact(ground_truth_artifact)

    os.remove(predictions_path)
    os.remove(ground_truth_path)


    metrics_df, fig_predictions_analysis, fig_metrics, fig_biggest_miss, (arso_predictions_df, model_predictions_df, actuals_df) = pef.plot_common(station_id, np.concatenate(all_actuals, axis=0), np.concatenate(all_predictions, axis=0), all_timestamps)
    # some annoying stuff fo wandb.Table
    arso_predictions_df.columns = arso_predictions_df.columns.astype(str)
    model_predictions_df.columns = model_predictions_df.columns.astype(str)
    actuals_df.columns = actuals_df.columns.astype(str)
    # Log the figures and metrics to wandb
    wandb.log({
        "predictions_analysis": wandb.Image(fig_predictions_analysis),
        "metrics_plot": wandb.Image(fig_metrics),
        "biggest_miss_plot": wandb.Image(fig_biggest_miss),
        "metrics_table": wandb.Table(dataframe=metrics_df),
        "arso_predictions_df": wandb.Table(dataframe=arso_predictions_df),
        "model_predictions_df": wandb.Table(dataframe=model_predictions_df),
        "actuals_df": wandb.Table(dataframe=actuals_df)
    })

    # Finish the wandb run
    wandb.finish()
    print(f"Run URL: {run.get_url()}")
    plt.close('all')
    return (arso_predictions_df, model_predictions_df, actuals_df)

brokens = []
for station_id in [#1060,1100,1140,1165,1220,1260,1300,1312,1335,1355,
                                            #2005,2010,2138,2140,2150,2160,2250,2372,2390,2420,2428,2432,2530,2600,2620,2640,2652,2667,2693,2719,2754,2830,2880,2900,
                                            #3014,3080,3105,3108,3115,3180,3200,3250,3260,3300,3320,3420,3465,3530,3550,3660,3725,3850,3900,
                                            #4025,4050,4095,4120,4155,4200,4209,4222,4230,4270,4298,4400,4430,4480,4515,4520,4570,4575,4626,4650,4671,4695,4706,4710,4750,4770,4791,4820,4828,4860,4960,4965,4969,4986,
                                            #5030,5078,5240,5270,5330,5425,5440,5479,5500,5540,5580,5770,5800,
                                            #5840,
                                            #5910,5940,
                                            #6020, 6060,6068,
                                            #6120,6140,6200,6210,6220,6240,6280,6300,6340,6415,6550,6630,6691,6720,6770,6790,6835,
                                            #7029,7060,7110,7160,7200,7220,7230,7235,7245,7272,7308,7340,7350,7380,7409,7440,7488,
                                            #8031,8060,8080,8180,
                                            #8270,8332,8350,8351,8450,8454,8478,8500,8545,8561,8565,8591,8601,8610,8630,8640,8660,8670,8680,8690,8700,8710,
                                            #9015,9030,9050,9077,9100,9108,9210,9240,9275,9280,9300
6200, ]:
    try:
        log_and_evaluate_pipeline(station_id, pipeline)
    except Exception as e:
        print(f"Error for station {station_id}: {e}")
        brokens.append(station_id)
        continue
print(brokens)


In [ ]:
feature_selector.n_features_in_, feature_selector.get_feature_names_out()

In [ ]:
arso_pred_df, model_pred_df, actuals_df = resDfs
arso_pred_df = arso_pred_df.set_index('index')
arso_pred_df.index = pd.to_datetime(arso_pred_df.index)
model_pred_df = model_pred_df.set_index('index')
model_pred_df.index = pd.to_datetime(model_pred_df.index)

actuals_df = actuals_df.set_index('index')
actuals_df.index = pd.to_datetime(actuals_df.index)


In [ ]:
actuals_df

In [ ]:
model_pred_df

In [ ]:
import ipywidgets as widgets
from IPython.display import display
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from datetime import datetime

def create_interactive_plots(arso_pred_df, model_pred_df, actuals_df):
    """
    Create interactive plots for comparing predictions across dates and hours.
    
    Args:
        arso_pred_df: DataFrame with ARSO predictions
        model_pred_df: DataFrame with model predictions
        actuals_df: DataFrame with actual values
    """
    def plot_predictions_for_date(date_idx, arso_pred_df, model_pred_df, actuals_df):
        """Plot predictions for a specific date from all three dataframes."""
        dates = actuals_df.index.unique()
        if date_idx >= len(dates):
            print(f"Date index {date_idx} out of range. Max index: {len(dates)-1}")
            return
        
        selected_date = dates[date_idx]
        
        arso_data = arso_pred_df.loc[selected_date] if selected_date in arso_pred_df.index else None
        model_data = model_pred_df.loc[selected_date] if selected_date in model_pred_df.index else None
        actual_data = actuals_df.loc[selected_date] if selected_date in actuals_df.index else None
        
        fig, ax = plt.subplots(figsize=(12, 6))
        hours = range(18)
        
        if actual_data is not None:
            ax.plot(hours, actual_data.values, 'k-', linewidth=2, label='Actual')
        if model_data is not None:
            ax.plot(hours, model_data.values, 'b-', linewidth=2, label='Model Prediction')
        if arso_data is not None:
            ax.plot(hours, arso_data.values, 'r-', linewidth=2, label='ARSO Prediction')
        
        ax.set_xlabel('Hours Ahead')
        ax.set_ylabel('Flow (m³/s)')
        ax.set_title(f'Predictions for {selected_date.strftime("%Y-%m-%d")}')
        ax.grid(True)
        ax.legend()
        
        plt.tight_layout()
        plt.show()
        return fig

    def plot_predictions_by_hour(hour, arso_pred_df, model_pred_df, actuals_df):
        """Plot all predictions for a specific hour ahead"""
        if hour < 0 or hour > 17:
            print(f"Hour must be between 0 and 17. Got {hour}")
            return
        
        arso_data = arso_pred_df[hour]
        model_data = model_pred_df[hour]
        actual_data = actuals_df[hour]
        
        fig, ax = plt.subplots(figsize=(12, 6))
        dates = actuals_df.index.unique()
        
        ax.plot(dates, actual_data, 'k-', linewidth=1, label='Actual')
        ax.plot(dates, model_data, 'b-', linewidth=1, label='Model Prediction')
        ax.plot(dates, arso_data, 'r-', linewidth=1, label='ARSO Prediction')
        
        ax.set_xlabel('Date')
        ax.set_ylabel('Flow (m³/s)')
        ax.set_title(f'Predictions {hour} Hours Ahead')
        ax.grid(True)
        ax.legend()
        
        fig.autofmt_xdate()
        plt.tight_layout()
        plt.show()
        return fig

    # Create date selection widgets
    num_dates = len(actuals_df.index.unique())
    date_slider = widgets.IntSlider(
        value=0,
        min=0,
        max=num_dates-1,
        step=1,
        description='Date:',
        continuous_update=False,
        layout=widgets.Layout(width='600px')
    )
    
    date_dropdown = widgets.Dropdown(
        options=[(d.strftime('%Y-%m-%d'), i) for i, d in enumerate(actuals_df.index.unique())],
        description='Select date:',
        layout=widgets.Layout(width='300px')
    )
    
    widgets.jslink((date_slider, 'value'), (date_dropdown, 'index'))
    
    date_output = widgets.Output()
    
    def update_date_plot(change):
        date_output.clear_output(wait=True)
        with date_output:
            plot_predictions_for_date(change['new'], arso_pred_df, model_pred_df, actuals_df)
    
    date_slider.observe(update_date_plot, names='value')
    
    with date_output:
        plot_predictions_for_date(date_slider.value, arso_pred_df, model_pred_df, actuals_df)
    
    # Create hour selection widgets
    hour_slider = widgets.IntSlider(
        value=0,
        min=0,
        max=17,
        step=1,
        description='Hour ahead:',
        continuous_update=False,
        layout=widgets.Layout(width='600px')
    )
    
    hour_output = widgets.Output()
    
    def update_hour_plot(change):
        hour_output.clear_output(wait=True)
        with hour_output:
            plot_predictions_by_hour(change['new'], arso_pred_df, model_pred_df, actuals_df)
    
    hour_slider.observe(update_hour_plot, names='value')
    
    with hour_output:
        plot_predictions_by_hour(hour_slider.value, arso_pred_df, model_pred_df, actuals_df)
    
    # Display all widgets
    display(widgets.VBox([
        widgets.HBox([date_slider, date_dropdown]),
        date_output,
        hour_slider,
        hour_output
    ]))

# Example usage:
create_interactive_plots(arso_pred_df, model_pred_df, actuals_df)







In [ ]:
from sklearn.linear_model import Lasso
from sklearn.model_selection import GridSearchCV

class LogLassoPipeline(Pipeline):
    def __init__(self, grid_search=None, *args, **kwargs):
        super().__init__(steps=grid_search,*args, **kwargs)
        self.grid_search = grid_search

    def fit(self, X, y):
        # If we have a grid_search object, use it to find best parameters
        if self.grid_search is not None:
            self.grid_search.fit(X, y)
            best_alpha = self.grid_search.best_params_["regressor__alpha"]
            best_score = self.grid_search.best_score_
            
            # Log the best parameters to wandb
            wandb.log({
                "best_alpha": best_alpha,
                "best_neg_mse": best_score
            })

            
        # Fit the pipeline with the best parameters
        return self
        
    def predict(self, X):
        return self.grid_search.predict(X)

# Create pipeline with Lasso regression
pipeline = Pipeline([
    #('scaler', StandardScaler()), ofc this has no effect
    ('regressor', Lasso(max_iter=10000, selection='random', random_state=42))
])

# Define alpha values to try
alphas = [0.001, 0.01, 0.1, 1.0, 10.0]

# Create grid search with cross-validation
param_grid = {'regressor__alpha': alphas}
grid_search = GridSearchCV(pipeline, param_grid, cv=3, scoring='neg_mean_squared_error')

lasso_pipeline = LogLassoPipeline(grid_search=grid_search)

# Fit grid search and log best parameters
method_name = 'lasso-regression'
log_and_evaluate_pipeline(5479, lasso_pipeline, method_name)

In [ ]:
from sklearn.linear_model import Lasso
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import GridSearchCV, KFold
import numpy as np
from torch.utils.data import DataLoader
import pandas as pd
import matplotlib.pyplot as plt

cv_train_losses = []
cv_test_losses = []
alpha_performance = {}  # Dictionary to store performance metrics for each alpha

for train_dataset, test_dataset in cv_splits:
    dataloader_train = DataLoader(train_dataset, batch_size=len(train_dataset))
    dataloader_test = DataLoader(test_dataset, batch_size=len(test_dataset), shuffle=False)

    # get first batch from dataloader
    batch = next(iter(dataloader_train))
    X_train, y_train, prediction_timestamp_train = batch['features'].numpy(), batch['target'].numpy(), list(pd.to_datetime(batch['prediction_timestamp'].numpy(), unit='s'))

    batch = next(iter(dataloader_test))
    X_test, y_test, prediction_timestamp_test = batch['features'].numpy(), batch['target'].numpy(), list(pd.to_datetime(batch['prediction_timestamp'].numpy(), unit='s'))

    # Check that test dates are sorted by date
    # Convert timestamps to numpy array for easier comparison
    prediction_timestamps_array = np.array(prediction_timestamp_test)

    # Check if timestamps are in ascending order
    if not np.all(np.diff(prediction_timestamps_array) >= pd.to_timedelta(0)):
        raise ValueError("Test dates are not sorted by date.")
    
    # Define the parameter grid for Lasso regression
    # Define a range of alpha values for Lasso regularization
    alphas = np.logspace(-2,1, 4)  # [0.001, 0.01, 0.1, 1, 10, 100]
    print(alphas)
    param_grid = {
        'regressor__alpha': alphas
    }
    
    # Create a pipeline with Lasso regression
    lasso_pipeline = Pipeline([
        ('regressor', Lasso(max_iter=10000, random_state=42, selection='random'))
    ])
    
    # Set up nested cross-validation
    inner_cv = KFold(n_splits=3, shuffle=True, random_state=42)
    
    # Create GridSearchCV with inner cross-validation
    grid_search = GridSearchCV(
        lasso_pipeline, 
        param_grid, 
        cv=inner_cv,
        scoring='neg_mean_squared_error',
        n_jobs=-1
    )
    
    # Fit the grid search to find the best alpha
    grid_search.fit(X_train, y_train)
    
    # Get the best model
    best_alpha = grid_search.best_params_['regressor__alpha']
    best_model = grid_search.best_estimator_
    
    print(f"Best alpha: {best_alpha}")
    
    # Calculate training loss with the best model
    train_pred = best_model.predict(X_train)
    train_loss = mean_squared_error(y_train, train_pred)
    cv_train_losses.append(train_loss)
    print(f"Train MSE: {train_loss:.4f}")
    
    # Calculate test loss with the best model
    y_pred = best_model.predict(X_test)
    test_loss = mean_squared_error(y_test, y_pred)
    cv_test_losses.append(test_loss)
    print(f"Test MSE: {test_loss:.4f}")
    
    # Store performance for each alpha
    for alpha in alphas:
        if alpha not in alpha_performance:
            alpha_performance[alpha] = {'train_mse': [], 'test_mse': []}
        
        # Create and fit a model with this specific alpha
        alpha_model = Pipeline([
            ('regressor', Lasso(alpha=alpha, max_iter=10000, random_state=42, selection='random'))
        ])
        alpha_model.fit(X_train, y_train)
        
        # Calculate and store metrics
        alpha_train_pred = alpha_model.predict(X_train)
        alpha_test_pred = alpha_model.predict(X_test)
        alpha_train_mse = mean_squared_error(y_train, alpha_train_pred)
        alpha_test_mse = mean_squared_error(y_test, alpha_test_pred)
        
        alpha_performance[alpha]['train_mse'].append(alpha_train_mse)
        alpha_performance[alpha]['test_mse'].append(alpha_test_mse)

print(f"CV Train Loss: {np.mean(cv_train_losses):.4f}")
print(f"CV Test Loss: {np.mean(cv_test_losses):.4f}")

# Plot performance per alpha
plt.figure(figsize=(10, 6))
train_means = [np.mean(alpha_performance[alpha]['train_mse']) for alpha in alphas]
test_means = [np.mean(alpha_performance[alpha]['test_mse']) for alpha in alphas]

plt.semilogx(alphas, train_means, 'o-', label='Train MSE')
plt.semilogx(alphas, test_means, 'o-', label='Test MSE')
plt.xlabel('Alpha (log scale)')
plt.ylabel('Mean Squared Error')
plt.title('Lasso Regression Performance by Alpha Value')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.pipeline import Pipeline


cv_train_losses = []
cv_test_losses = []

for train_dataset, test_dataset in cv_splits:
    dataloader_train = DataLoader(train_dataset, batch_size=len(train_dataset))
    dataloader_test = DataLoader(test_dataset, batch_size=len(test_dataset), shuffle=False)

    # get first batch from dataloader
    batch = next(iter(dataloader_train))
    X_train, y_train, prediction_timestamp_train = batch['features'].numpy(), batch['target'].numpy(), list(pd.to_datetime(batch['prediction_timestamp'].numpy(), unit='s'))

    batch = next(iter(dataloader_test))
    X_test, y_test, prediction_timestamp_test = batch['features'].numpy(), batch['target'].numpy(), list(pd.to_datetime(batch['prediction_timestamp'].numpy(), unit='s'))

    # Check that test dates are sorted by date
    # Convert timestamps to numpy array for easier comparison
    prediction_timestamps_array = np.array(prediction_timestamp_test)

    # Check if timestamps are in ascending order
    if not np.all(np.diff(prediction_timestamps_array) >= pd.to_timedelta(0)):
        raise ValueError("Test dates are not sorted by date.")
    
    #print("Data shapes ",X_train.shape, y_train.shape, X_test.shape, y_test.shape)

    pipeline = Pipeline([
        #('scaler', StandardScaler()), ofc this has no effect
        ('regressor', LinearRegression())
    ])
    pipeline.fit(X_train, y_train)
    
    test_loss = mean_squared_error(y_train, pipeline.predict(X_train))
    print(f"Train MSE: {test_loss:.4f}")
    cv_train_losses.append(test_loss)

    y_pred = pipeline.predict(X_test)
    mse = mean_squared_error(y_pred, y_test)
    print(f"Test MSE: {mse}")
    cv_test_losses.append(mse)

print(f"CV Train Loss: {np.mean(cv_train_losses):.4f}")
print(f"CV Test Loss: {np.mean(cv_test_losses):.4f}")

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.pipeline import Pipeline
import numpy as np
import pandas as pd


cv_train_losses = []
cv_test_losses = []
all_y_test = []
all_y_pred = []

for train_dataset, test_dataset in cv_splits:
    dataloader_train = DataLoader(train_dataset, batch_size=len(train_dataset))
    dataloader_test = DataLoader(test_dataset, batch_size=len(test_dataset), shuffle=False)

    # get first batch from dataloader
    batch = next(iter(dataloader_train))
    X_train, y_train, prediction_timestamp_train = batch['features'].numpy(), batch['target'].numpy(), list(pd.to_datetime(batch['prediction_timestamp'].numpy(), unit='s'))

    batch = next(iter(dataloader_test))
    X_test, y_test, prediction_timestamp_test = batch['features'].numpy(), batch['target'].numpy(), list(pd.to_datetime(batch['prediction_timestamp'].numpy(), unit='s'))

    # Check that test dates are sorted by date
    # Convert timestamps to numpy array for easier comparison
    prediction_timestamps_array = np.array(prediction_timestamp_test)

    # Check if timestamps are in ascending order
    if not np.all(np.diff(prediction_timestamps_array) >= pd.to_timedelta(0)):
        raise ValueError("Test dates are not sorted by date.")
    
    #print("Data shapes ",X_train.shape, y_train.shape, X_test.shape, y_test.shape)

    pipeline = Pipeline([
        #('scaler', StandardScaler()), ofc this has no effect
        ('regressor', LinearRegression())
    ])
    pipeline.fit(X_train, y_train)
    
    test_loss = mean_squared_error(y_train, pipeline.predict(X_train))
    print(f"Train MSE: {test_loss:.4f}")
    cv_train_losses.append(test_loss)

    y_pred = pipeline.predict(X_test)
    mse = mean_squared_error(y_pred, y_test)
    print(f"Test MSE: {mse}")
    cv_test_losses.append(mse)
    
    # Store actual and predicted values for later analysis
    all_y_test.append(y_test)
    all_y_pred.append(y_pred)

print(f"CV Train Loss: {np.mean(cv_train_losses):.4f}")
print(f"CV Test Loss: {np.mean(cv_test_losses):.4f}")

# Evaluate results based on the height of values
# Combine all test values and predictions
all_y_test_combined = np.vstack(all_y_test)  # Shape will be (total_samples, 18)
all_y_pred_combined = np.vstack(all_y_pred)  # Shape will be (total_samples, 18)

# For each sample, find the maximum value across all 18 dimensions
max_per_sample = np.max(all_y_test_combined, axis=1)  # Get max value for each sample

# Find the overall min and max to create buckets
overall_max = np.max(max_per_sample)
overall_min = np.min(max_per_sample)

# Create 10 buckets based on the max value in each sample
bucket_edges = np.linspace(overall_min, overall_max, 5)
bucket_errors = []
bucket_counts = []

print("\nError analysis by maximum value in each sample:")
print("----------------------------------------------")
for i in range(4):
    lower = bucket_edges[i]
    upper = bucket_edges[i+1]
    
    # Find samples where the maximum value falls in this range
    sample_max_values = np.max(all_y_test_combined, axis=1)
    indices = np.where((sample_max_values > lower) & (sample_max_values <= upper))[0]
    
    if len(indices) > 0:
        # Calculate error for this bucket across all dimensions
        bucket_y_test = all_y_test_combined[indices]
        bucket_y_pred = all_y_pred_combined[indices]
        
        # Compute MSE across all dimensions for these samples
        bucket_mse = mean_squared_error(bucket_y_test, bucket_y_pred)
        
        bucket_errors.append(bucket_mse)
        bucket_counts.append(len(indices))
        
        print(f"Max value range [{lower:.2f}, {upper:.2f}): Count: {len(indices)}, MSE: {bucket_mse:.4f}")
    else:
        print(f"Max value range [{lower:.2f}, {upper:.2f}): No samples")
        bucket_errors.append(0)
        bucket_counts.append(0)

# Print summary
print("\nSummary: Bucket with highest error:")
if len(bucket_errors) > 0 and max(bucket_counts) > 0:
    max_error_idx = np.argmax([err if count > 0 else 0 for err, count in zip(bucket_errors, bucket_counts)])
    print(f"Max value range [{bucket_edges[max_error_idx]:.2f}, {bucket_edges[max_error_idx+1]:.2f}) has highest MSE: {bucket_errors[max_error_idx]:.4f}")

In [ ]:
# Some Feature Selection Code
"""from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
import numpy as np

# Initialize model
model = LinearRegression()

# Track losses for each number of features
train_losses = []
test_losses = []
selected_features_list = []

# Try different numbers of features
for n_features in range(1, X_train.shape[1] + 1):
    # Create selector and pipeline
    selector = SequentialFeatureSelector(model, n_features_to_select=n_features, direction='forward')
    pipeline = Pipeline([
        ('feature_selection', selector),
        ('regressor', model)
    ])
    
    # Fit pipeline
    pipeline.fit(X_train, y_train)
    
    # Get selected features
    selected_features = selector.get_support()
    selected_feature_indices = np.where(selected_features)[0]
    selected_features_list.append(selected_feature_indices)
    
    # Calculate losses
    train_pred = pipeline.predict(X_train)
    test_pred = pipeline.predict(X_test)
    train_loss = mean_squared_error(y_train, train_pred)
    test_loss = mean_squared_error(y_test, test_pred)
    
    train_losses.append(train_loss)
    test_losses.append(test_loss)
    
    print(f"\nNumber of features: {n_features}")
    print(f"Selected features: {selected_feature_indices}")
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Test Loss: {test_loss:.4f}")

# Find optimal number of features
optimal_n_features = np.argmin(test_losses) + 1
print(f"\nOptimal number of features: {optimal_n_features}")
print(f"Selected features at optimal point: {selected_features_list[optimal_n_features-1]}")
print(f"Best test loss: {min(test_losses):.4f}")
"""

In [ ]:
from sklearn.linear_model import Lasso
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import GridSearchCV, KFold
import numpy as np
from torch.utils.data import DataLoader
import pandas as pd
import matplotlib.pyplot as plt

cv_train_losses = []
cv_test_losses = []
alpha_performance = {}  # Dictionary to store performance metrics for each alpha

for train_dataset, test_dataset in cv_splits:
    dataloader_train = DataLoader(train_dataset, batch_size=len(train_dataset))
    dataloader_test = DataLoader(test_dataset, batch_size=len(test_dataset), shuffle=False)

    # get first batch from dataloader
    batch = next(iter(dataloader_train))
    X_train, y_train, prediction_timestamp_train = batch['features'].numpy(), batch['target'].numpy(), list(pd.to_datetime(batch['prediction_timestamp'].numpy(), unit='s'))

    batch = next(iter(dataloader_test))
    X_test, y_test, prediction_timestamp_test = batch['features'].numpy(), batch['target'].numpy(), list(pd.to_datetime(batch['prediction_timestamp'].numpy(), unit='s'))

    # Check that test dates are sorted by date
    # Convert timestamps to numpy array for easier comparison
    prediction_timestamps_array = np.array(prediction_timestamp_test)

    # Check if timestamps are in ascending order
    if not np.all(np.diff(prediction_timestamps_array) >= pd.to_timedelta(0)):
        raise ValueError("Test dates are not sorted by date.")
    
    # Define the parameter grid for Lasso regression
    # Define a range of alpha values for Lasso regularization
    alphas = np.logspace(-2,1, 4)  # [0.001, 0.01, 0.1, 1, 10, 100]
    print(alphas)
    param_grid = {
        'regressor__alpha': alphas
    }
    
    # Create a pipeline with Lasso regression
    lasso_pipeline = Pipeline([
        ('regressor', Lasso(max_iter=10000, random_state=42, selection='random'))
    ])
    
    # Set up nested cross-validation
    inner_cv = KFold(n_splits=3, shuffle=True, random_state=42)
    
    # Create GridSearchCV with inner cross-validation
    grid_search = GridSearchCV(
        lasso_pipeline, 
        param_grid, 
        cv=inner_cv,
        scoring='neg_mean_squared_error',
        n_jobs=-1
    )
    
    # Fit the grid search to find the best alpha
    grid_search.fit(X_train, y_train)
    
    # Get the best model
    best_alpha = grid_search.best_params_['regressor__alpha']
    best_model = grid_search.best_estimator_
    
    print(f"Best alpha: {best_alpha}")
    
    # Calculate training loss with the best model
    train_pred = best_model.predict(X_train)
    train_loss = mean_squared_error(y_train, train_pred)
    cv_train_losses.append(train_loss)
    print(f"Train MSE: {train_loss:.4f}")
    
    # Calculate test loss with the best model
    y_pred = best_model.predict(X_test)
    test_loss = mean_squared_error(y_test, y_pred)
    cv_test_losses.append(test_loss)
    print(f"Test MSE: {test_loss:.4f}")
    
    # Store performance for each alpha
    for alpha in alphas:
        if alpha not in alpha_performance:
            alpha_performance[alpha] = {'train_mse': [], 'test_mse': []}
        
        # Create and fit a model with this specific alpha
        alpha_model = Pipeline([
            ('regressor', Lasso(alpha=alpha, max_iter=10000, random_state=42, selection='random'))
        ])
        alpha_model.fit(X_train, y_train)
        
        # Calculate and store metrics
        alpha_train_pred = alpha_model.predict(X_train)
        alpha_test_pred = alpha_model.predict(X_test)
        alpha_train_mse = mean_squared_error(y_train, alpha_train_pred)
        alpha_test_mse = mean_squared_error(y_test, alpha_test_pred)
        
        alpha_performance[alpha]['train_mse'].append(alpha_train_mse)
        alpha_performance[alpha]['test_mse'].append(alpha_test_mse)

print(f"CV Train Loss: {np.mean(cv_train_losses):.4f}")
print(f"CV Test Loss: {np.mean(cv_test_losses):.4f}")

# Plot performance per alpha
plt.figure(figsize=(10, 6))
train_means = [np.mean(alpha_performance[alpha]['train_mse']) for alpha in alphas]
test_means = [np.mean(alpha_performance[alpha]['test_mse']) for alpha in alphas]

plt.semilogx(alphas, train_means, 'o-', label='Train MSE')
plt.semilogx(alphas, test_means, 'o-', label='Test MSE')
plt.xlabel('Alpha (log scale)')
plt.ylabel('Mean Squared Error')
plt.title('Lasso Regression Performance by Alpha Value')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
y_pred.shape, (y_pred[y_pred<0]).shape

In [ ]:
# get linear regression coefficients
coefficients = pipeline.named_steps['regressor'].coef_
# get linear regression intercept
intercept = pipeline.named_steps['regressor'].intercept_


In [ ]:
import importlib
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import root_mean_squared_error 
import plotting_and_evaluation_functions as pef
importlib.reload(pef)

pef.plot_common(4200, y_test, pipeline.predict(X_test), prediction_timestamp_test)


In [ ]:
import importlib
import ARSO_model
importlib.reload(ARSO_model)


arso_model = ARSO_model.ARSO_model(4200)
arso_prediction_dates = arso_model.get_valid_dates()
# Find common dates between ARSO predictions and the test timestamps
evaluation_dates = sorted(list(set(arso_prediction_dates).intersection(set(prediction_timestamp_test))))

arso_model_predictions = arso_model.get_prediction_for_dates(evaluation_dates)
print(arso_model_predictions)
y_test_filtered = y_test[np.isin(prediction_timestamp_test, evaluation_dates)]

plot_predictions_analysis(y_test_filtered, arso_model_predictions)



In [ ]:
plot_predictions_analysis(y_test_filtered, pipeline.predict(X_test)[np.isin(prediction_timestamp_test, evaluation_dates)])

In [ ]:
# Calculate NS (Nash-Sutcliffe efficiency), Per (Pearson Correlation Coefficient), and dQs (difference in predicted vs ground truth) per hour
def calculate_metrics(y_true, y_pred):
    # Calculate Nash-Sutcliffe efficiency
    numerator = np.sum((y_true - y_pred) ** 2)
    denominator = np.sum((y_true - np.mean(y_true)) ** 2)
    ns = 1 - (numerator / denominator) if denominator != 0 else np.nan

    # Calculate Pearson Correlation Coefficient
    # Using numpy's corrcoef function which returns the Pearson correlation coefficient
    correlation = np.corrcoef(y_true, y_pred)[0, 1]
    per = correlation  # This is already the Pearson correlation coefficient

    # Calculate dQs (difference in predicted vs ground truth)
    dQs = np.sum(y_pred) / np.sum(y_true)

    return ns, per, dQs

import pandas as pd

def plot_metrics(y_test, y_pred, show_plots=True, title=None):
    # Initialize lists to store metrics
    ns_per_hour = []
    per_corr_per_hour = []
    dQs_per_hour = []

    # Extract num_hours from the shape of y_test
    num_hours = y_test.shape[1]

    # Calculate metrics for each hour
    for hour in range(num_hours):
        y_true_hour = y_test[:, hour]
        y_pred_hour = y_pred[:, hour]
        
        ns, per, dQs = calculate_metrics(y_true_hour, y_pred_hour)
        ns_per_hour.append(ns)
        per_corr_per_hour.append(per)
        dQs_per_hour.append(dQs)

    # Convert to numpy arrays for easier handling
    ns_per_hour = np.array(ns_per_hour)
    per_corr_per_hour = np.array(per_corr_per_hour)
    dQs_per_hour = np.array(dQs_per_hour)

    # Create a DataFrame to return the metrics
    metrics_df = pd.DataFrame({
        'Hour': np.arange(1, num_hours + 1),
        'Nash-Sutcliffe Efficiency': ns_per_hour,
        'Pearson Correlation Coefficient': per_corr_per_hour,
        'dQs': dQs_per_hour
    })

    # Define maximum values for capping
    max_ns = 1.0
    max_per = 1.0

    # Plotting the metrics if show_plots is True
    if show_plots:
        plt.figure(figsize=(12, 6))

        # Nash-Sutcliffe Efficiency
        plt.subplot(3, 1, 1)
        plt.plot(ns_per_hour, marker='o', linestyle='-', label='Nash-Sutcliffe Efficiency')
        plt.title('Nash-Sutcliffe Efficiency per Hour')
        plt.xlabel('Hour')
        plt.ylabel('NSE')
        plt.xticks(ticks=np.arange(num_hours), labels=np.arange(1, num_hours + 1))  # Start hour from 1
        plt.ylim(min(0.6, np.floor(np.min(ns_per_hour) * 10) / 10), min(max_ns, np.ceil(np.max(ns_per_hour) * 10) / 10) + 0.04)  # Dynamic upper limit capped
        plt.grid()

        # Pearson Correlation Coefficient
        plt.subplot(3, 1, 2)
        plt.plot(per_corr_per_hour, marker='o', linestyle='-', label='Pearson Correlation Coefficient', color='orange')
        plt.title('Pearson Correlation Coefficient per Hour')
        plt.xlabel('Hour')
        plt.ylabel('Pearson Correlation Coefficient')
        plt.xticks(ticks=np.arange(num_hours), labels=np.arange(1, num_hours + 1))  # Start hour from 1
        plt.ylim(min(0.6, np.floor(np.min(per_corr_per_hour) * 10) / 10), min(max_per, np.ceil(np.max(per_corr_per_hour) * 10) / 10)+ 0.04)  # Dynamic upper limit capped
        plt.grid()

        # Difference in Predicted vs Ground Truth (dQs)
        plt.subplot(3, 1, 3)
        plt.plot(dQs_per_hour, marker='o', linestyle='-', label='Difference in Predicted vs Ground Truth', color='green')
        plt.title('Difference in Predicted vs Ground Truth (dQs) per Hour')
        plt.xlabel('Hour')
        plt.ylabel('dQs')
        plt.xticks(ticks=np.arange(num_hours), labels=np.arange(1, num_hours + 1))  # Start hour from 1
        plt.ylim( np.floor(np.min(dQs_per_hour) * 100) / 100,  np.ceil(np.max(dQs_per_hour) * 100) / 100)  # Dynamic upper limit capped
        plt.grid()

        if title:
            plt.suptitle(title)  # Add title to the entire figure if provided

        plt.tight_layout()  # Adjust layout to prevent clipping of tick-labels
        plt.show()

    return metrics_df

# Call the function and store the returned DataFrame
metrics_dataframe = plot_metrics(y_test, y_pred, title="Linear Regression Model")


In [ ]:
plot_metrics(y_test_filtered, arso_model_predictions, title="ARSO Model")
plot_metrics(y_test_filtered, pipeline.predict(X_test)[np.isin(prediction_timestamp_test, evaluation_dates)], title="Linear Regression Model")